<div style="background-color: #ffffff; color: #000000; padding: 10px;">
<div style="display: flex; justify-content: space-between; align-items: center; background-color: #ffffff; color: #000000; padding: 10px;">
    <img src="../../00_aisc/img/logo_aisc_150dpi.png" height="80" style="margin-right: auto;" alt="Logo of the AI Service Center Berlin-Brandenburg.">
    <img src="../../00_aisc/img/logo_bmftr_de.png" height="150" style="margin-left: auto;" alt="Logo of the German Federal Ministry of Research, Technology and Space: Gefoerdert vom Bundesministerium fuer Forschung, Technologie und Raumfahrt.">
</div>
<h1> Workshop: Rapid Prototyping via Vibe Coding
<h2> Part 1 &mdash; docling-toolkit
</div>

<div style="background-color: #f6a800; color: #ffffff; padding: 10px;">
    <h2> Part 1 &mdash; docling-toolkit</h2>
</div>

**docling-toolkit** is a Claude Code plugin that wraps IBM's [Docling](https://github.com/docling-project/docling) document-processing library into skills, commands, and agents Claude can invoke directly.

The pain it solves: every vibe-coded prototype that touches PDFs eventually needs a parser. Writing that parser yourself is a rabbit hole (multi-column layouts, tables, OCR, citations). Docling already handles it. The plugin teaches Claude *how* to use Docling, so you can say *"extract these PDFs into structured JSONL"* and get a working script in one turn.

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>1. What is Docling?</h3>
</div>

Docling is an open-source document-processing library developed at IBM Research and donated to the Linux Foundation AI & Data initiative.

Four things that make it a good default:

- **Structure-aware chunking** &mdash; preserves document hierarchy (sections, paragraphs, tables, figures) instead of splitting blindly by character count.
- **Rich metadata** &mdash; page numbers, section titles, layout information, bounding boxes.
- **Granite model support** &mdash; enhanced processing for scanned documents and complex layouts.
- **Enterprise-grade reliability** &mdash; battle-tested at IBM; 42k+ GitHub stars, 1.5M monthly downloads at the time of writing.

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>2. Why it matters for rapid prototyping</h3>
</div>

When you're sketching a prototype that needs to answer questions over a bundle of PDFs (RAG demos, policy analysis, internal-docs search, compliance diffs), you want to spend your time on the *interesting* part &mdash; retrieval, ranking, UI. Docling lets you offload the parsing to a battle-tested pipeline in one command and move on.

The output format is citation-rich JSONL: every chunk carries `file`, `page_start`, `page_end`, `section_title`, and the raw text. That shape plugs directly into most embedding pipelines without custom glue.

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>3. What the plugin adds</h3>
</div>

The `docling-toolkit` plugin bundles three categories of capabilities:

#### Skills (AI-invoked automatically)

Claude will reach for these when you ask the right-shaped question:

| Skill | Triggers on topics like... |
|-------|---------------------------|
| `docling-fundamentals` | installation, basic usage, *"when should I use Docling vs. PyPDF2?"* |
| `docling-chunking` | chunking strategy, `HybridChunker` vs. `HierarchicalChunker`, metadata extraction |
| `docling-advanced` | Granite model, scanned PDFs, complex layouts, performance tuning |

#### Commands (you invoke)

Slash commands for predictable scaffolding:

| Command | What it does |
|---------|--------------|
| `/docling-init-project <name>` | Creates a full project skeleton (`data/raw/`, `data/processed/`, `extracts/`, `scripts/`, config, `.env.example`). |
| `/docling-scaffold-processor <name> --input-types pdf,html` | Generates a production-ready Python script with `HybridChunker`, CLI args, progress bars, error handling, and JSONL output. |
| `/docling-validate-extracts` | Reviews the emitted JSONL for missing metadata, empty chunks, and encoding issues. |

#### Agents (delegated subagents)

Two specialists you can invoke by asking Claude to *"use a subagent"*:

| Agent | When to ask for it |
|-------|-------------------|
| `docling-advisor` | Designing the extraction workflow: which chunker, Granite or not, how to split, what metadata. |
| `script-advisor` | Debugging or customising a script the scaffolder has already produced. |

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>4. Install (reference)</h3>
</div>

The plugin is already installed in your Claude Code environment. For reference, the install flow for a fresh machine is:

```bash
# Prereqs
# 1) Claude Code installed (you have this).
# 2) Python 3.11+.
# 3) Docling itself:
uv add docling
# or:
pip install docling
```

Then the plugin (installed from a local path in this example):

```bash
claude plugin install ./docling-toolkit --scope user
claude plugin list   # verify "docling-toolkit" appears
```

If installing from a git marketplace, the flow is:

```bash
claude plugin marketplace add <org>/<repo>
claude plugin install docling-toolkit@<marketplace-name>
```

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>5. Quick-start walkthrough</h3>
</div>

End-to-end, from empty directory to validated extracts, in four Claude Code turns.

#### 1. Scaffold the project

```text
/docling-init-project my-extraction
```

Expected result: Claude creates this tree in the current directory:

```
my-extraction/
+-- README.md
+-- config/
|   +-- docling-config.yaml
+-- data/
|   +-- raw/              # drop PDFs / HTML here
|   +-- processed/
+-- extracts/             # Docling output (JSONL)
+-- scripts/
|   +-- process_documents.py
|   +-- validate_extracts.py
+-- logs/
+-- .env.example
```

#### 2. Scaffold the processor script

```text
/docling-scaffold-processor process_documents --input-types pdf,html
```

Claude generates (or updates) `scripts/process_documents.py` with:

- Docling `HybridChunker` integration.
- CLI args: `--input-dir`, `--output-file`, `--granite`.
- Progress tracking, error handling, structured logging.
- JSONL output with rich metadata per chunk.

#### 3. Run the extraction

Drop PDFs into `data/raw/`, then:

```bash
uv run python scripts/process_documents.py \
    --input-dir data/raw \
    --output-file extracts/output.jsonl
```

For scanned or complex-layout documents, add `--granite` and expect the first run to take longer while the model downloads.

#### 4. Validate

```text
/docling-validate-extracts
```

This reads the JSONL, checks every record has the expected metadata fields, flags empty chunks, and prints a per-file summary.

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>6. When to reach for which agent</h3>
</div>

| Question shape | Agent |
|---------------|-------|
| *"Should I use `HybridChunker` or `HierarchicalChunker` for annual reports?"* | `docling-advisor` |
| *"The chunker is keeping tables glued to adjacent prose. How do I separate them?"* | `docling-advisor` |
| *"The scaffolded script crashes on page 17 of this PDF. Can you fix it?"* | `script-advisor` |
| *"I want the processor to also emit a parquet sidecar file."* | `script-advisor` |

Rule of thumb: **advisor** for *design*, **script-advisor** for *code*.

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>7. How this composes with the rest of Part 1</h3>
</div>

- Pair with [`context-mode`](05_context_mode.ipynb) when you're ingesting many large PDFs &mdash; the raw Docling intermediate output is noisy; context-mode keeps it out of Claude's main context.
- Pair with [`gepa`](04_gepa.ipynb) when you're iterating on the **prompt** a downstream RAG agent uses to reason over the Docling JSONL &mdash; GEPA optimises that prompt against a held-out evaluation set.